In [3]:
import monai

c:\Users\Lenovo\AppData\Local\Programs\Python\Python310\lib\site-packages\ignite\handlers\checkpoint.py:17: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


In [17]:
!python -m monai.bundle run --config_file "C:\Users\Lenovo\Desktop\Grad\Monai\radiology\vista3d\configs\inference.json"  




2025-03-30 22:42:58,789 - INFO - --- input summary of monai.bundle.scripts.run ---
2025-03-30 22:42:58,789 - INFO - > config_file: 'C:\\Users\\Lenovo\\Desktop\\Grad\\Monai\\radiology\\vista3d\\configs\\inference.json'
2025-03-30 22:42:58,789 - INFO - ---


2025-03-30 22:42:58,789 - INFO - Setting logging properties based on config: C:\Users\Lenovo\Desktop\Grad\Monai\radiology\vista3d\configs\logging.conf.
2025-03-30 22:43:01,241 - root - INFO - Restored all variables from C:/Users/Lenovo/Desktop/Grad/monai/radiology/vista3d/models/model.pt
2025-03-30 22:43:01,242 - ignite.engine.engine.Vista3dEvaluator - INFO - Engine run resuming from iteration 0, epoch 0 until 1 epochs
arrived
2025-03-30 22:43:09,905 INFO image_writer.py:197 - writing: C:\Users\Lenovo\Desktop\Grad\monai\radiology\vista3d\eval\1.2.840.113845.13.5681.3264321649.2854263722\1.2.840.113845.13.5681.3264321649.2854263722_trans.nii.gz
2025-03-30 22:43:10,709 - ignite.engine.engine.Vista3dEvaluator - INFO - Epoch[1] Complete.

c:\Users\Lenovo\AppData\Local\Programs\Python\Python310\lib\site-packages\ignite\handlers\checkpoint.py:17: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer
You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globa

# Automation

In [18]:
import json
import subprocess
import nibabel as nib
import matplotlib.pyplot as plt

def run_monai_inference(config_path, image_path, mode, label_index=None, point=None,point_labels=None):
    """
    Updates the `input_dict` field in the MONAI inference.json file and runs inference.

    Parameters:
    - config_path (str): Path to inference.json file.
    - image_path (str): Path to the input medical image.
    - label_index (int): Label index for segmentation.
    - point (list): List of [x, y, z] coordinates for the prompt.
    """
    
    # Load the current inference.json file
    with open(config_path, "r") as f:
        config = json.load(f)
    
    # Update the `input_dict` field
    if mode=="Interactive":
        config["input_dict"] = {
            "image": image_path,
            "label_prompt": [label_index],
            "points": point,  # Ensure it's a list of lists
            "point_labels": point_labels  # Assuming 1 is the default class label
        }
    else:
        config["input_dict"] = {
            "image": image_path,
            "label_prompt": [label_index],
        }


    # Save the updated config
    with open(config_path, "w") as f:
        json.dump(config, f, indent=4)

    # Run the MONAI inference command
    command = f'python -m monai.bundle run --config_file "{config_path}"'
    subprocess.run(command, shell=True, check=True)

    # plot the output


# Example usage
config_path = r"C:\Users\Lenovo\Desktop\Grad\monai_vista3d_0.5.4\monai_vista3d_0.5.4\configs\inference.json"
image_path = r"C:\Users\Lenovo\Desktop\Grad\Monai\radiology\vista3d\Data\patient_081_0000.nii.gz"
label_index = 133 # Example label
point = [[98, 92, 68]] # Example point
point_labels = [2]  # Example point labels

run_monai_inference(config_path, image_path, mode="Interactive", label_index=label_index, point=point,point_labels=point_labels)


In [19]:
# plot mask
import numpy as np
def plot_mask_overlay(image_path,mask_path):
    image=nib.load(image_path).get_fdata()
    mask = nib.load(mask_path).get_fdata()
    # mask unique values
    print(f"Unique values in mask: {np.unique(mask)}")
    # show only 0 and label_index
    label_index = 1
    mask[mask != label_index] = 0
    plt.imshow(image[:,:,22], cmap="gray")
    plt.imshow(mask[:,:,22], cmap="jet", alpha=0.5)

    plt.show()

mask_path = "eval/patient_081_0000/patient_082_0000_trans.nii.gz"
image_path = "Data/patient_081_0000.nii.gz"
plot_mask_overlay(image_path,mask_path)

FileNotFoundError: No such file or no access: 'eval/patient_081_0000/patient_082_0000_trans.nii.gz'

# compare sam2 with Vista


In [ ]:
def dice_score(pred, gt):
    # Flatten arrays
    pred = pred.flatten()
    gt = gt.flatten()
    
    # Calculate intersection and sums
    intersection = np.sum(pred * gt)
    return (2.0 * intersection) / (np.sum(pred) + np.sum(gt) + 1e-6)  # Add epsilon to avoid division by zero


def run_sam2(image_path, points):
    """Runs 3D SAM2 and returns segmentation mask path."""
    # Your SAM2 inference function here
    return "path_to_sam_output.nii.gz"  # Update with actual output path

def load_nifti(nifti_path):
    """Load NIfTI image as numpy array."""
    return nib.load(nifti_path).get_fdata()

def compare_models(image_path, gt_path, vista_config, label_index, points, point_labels, results_path):
    """Runs both models, computes Dice scores, and saves results."""
    gt = load_nifti(gt_path)
    
    vista_mask = load_nifti(run_monai_inference(vista_config, image_path, label_index, points, point_labels))
    sam_mask = load_nifti(run_sam2(image_path, points))
    
    dice_vista = dice_score(vista_mask, gt)
    dice_sam = dice_score(sam_mask, gt)
    
    results = {
        "image": image_path,
        "label_index": label_index,
        "dice_scores": {
            "vista3d": dice_vista,
            "sam2": dice_sam
        }
    }
    
    with open(results_path, "w") as f:
        json.dump(results, f, indent=4)
    
    print(f"Results saved to {results_path}")
    return results